In [ ]:
from google.colab import drive; drive.mount("/content/drive")
!pip install -q keras-hub onnx onnxruntime tf2onnx


Mounted at /content/drive


In [ ]:
import os, json, importlib.util, numpy as np, pandas as pd, tensorflow as tf, keras, keras_hub
from scipy.special import softmax
ROOT = "/content/drive/MyDrive/ticket_routing"

spec = importlib.util.spec_from_file_location("text", f"{ROOT}/src/text.py")
text_mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(text_mod)
clean_text, format_for_bert = text_mod.clean_text, text_mod.format_for_bert
def first_350(t): return " ".join(t.split()[:350])

keras.mixed_precision.set_global_policy("float32")                 # BEFORE building fresh
trained = keras.models.load_model(f"{ROOT}/models/bert_best.keras")
fresh = keras_hub.models.BertTextClassifier.from_preset(
    "bert_small_en_uncased", num_classes=11, preprocessor=None)
fresh.set_weights(trained.get_weights())                            # copy the trained weights

val = pd.read_parquet(f"{ROOT}/data/processed/val.parquet").sample(1000, random_state=42)   # VAL, not test
raw = val["Consumer complaint narrative"].tolist()
texts = [format_for_bert(first_350(clean_text(t))) for t in raw]

x = trained.preprocessor(tf.constant(texts))                        # token_ids, segment_ids, padding_mask
ds = tf.data.Dataset.from_tensor_slices(texts).batch(32)
probs_keras = softmax(np.asarray(trained.predict(ds, verbose=0), dtype="float32"), axis=1)
probs_fresh = softmax(np.asarray(fresh.predict(x, batch_size=32, verbose=0), dtype="float32"), axis=1)
print(f"float32 copy vs trained, argmax match: {(probs_keras.argmax(1) == probs_fresh.argmax(1)).mean():.1%}")

# save the "truth" for the local check
vocab = trained.preprocessor.tokenizer.get_vocabulary()
with open(f"{ROOT}/models/bert_vocab.txt", "w") as f:
    f.write("\n".join(vocab) + "\n")
np.save(f"{ROOT}/models/ref_token_ids.npy", np.asarray(x["token_ids"]))
np.save(f"{ROOT}/models/ref_probs.npy", probs_keras)
json.dump(raw, open(f"{ROOT}/models/ref_raw_texts.json", "w"))
print("vocab size:", len(vocab))                                    # expect 30522


/usr/local/lib/python3.13/dist-packages/keras/src/saving/serialization_lib.py:749: UserWarning: `compile()` was not called as part of model loading because the model's `compile()` method is custom. All subclassed Models that have `compile()` overridden should also override `get_compile_config()` and `compile_from_config(config)`. Alternatively, you can call `compile()` manually after loading.
  instance.compile_from_config(compile_config)
/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'loss_scale_optimizer', because it has 4 variables whereas the saved optimizer has 152 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 148 variables. 
  saveable.load_own_variables(weights_store.get(inner_pa

100%|██████████| 455/455 [00:00<00:00, 612kB/s]


100%|██████████| 110M/110M [00:02<00:00, 42.6MB/s]


float32 copy vs trained, argmax match: 100.0%
vocab size: 30522


In [ ]:
onnx_path = f"{ROOT}/models/bert.onnx"
fresh.export(onnx_path, format="onnx")
print("Route 1 ✅ | ONNX MB:", round(os.path.getsize(onnx_path) / 1e6, 1))


/usr/local/lib/python3.13/dist-packages/keras/src/export/tf2onnx_lib.py:20: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


Saved artifact at '/content/drive/MyDrive/ticket_routing/models/bert.onnx'.
Route 1 ✅ | ONNX MB: 127.8


In [ ]:
fresh.export("/content/bert_savedmodel", format="tf_saved_model")
!python -m tf2onnx.convert --saved-model /content/bert_savedmodel --output "{onnx_path}" --opset 17
print("Route 2 ✅ | ONNX MB:", round(os.path.getsize(onnx_path) / 1e6, 1))


Saved artifact at '/content/bert_savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): Dict[['token_ids', TensorSpec(shape=(None, None), dtype=tf.int32, name='token_ids')], ['segment_ids', TensorSpec(shape=(None, None), dtype=tf.int32, name='segment_ids')], ['padding_mask', TensorSpec(shape=(None, None), dtype=tf.int32, name='padding_mask')]]
Output Type:
  TensorSpec(shape=(None, 11), dtype=tf.float32, name=None)
Captures:
  132591818975440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132591818976016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132591818974864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132591818976784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132591818976400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132591818975632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132591818977360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132591818975248: TensorSpec(shape=